# GNSS and Coordinate Frames

This notebook covers the coordinate frame theory and transformations needed to feed GNSS data into an Extended Kalman Filter (EKF) for state estimation.

The math follows *Principles of GNSS, Inertial, and Multi-Sensor Integrated Navigation Systems*, specifically Chapter 2. Equation numbers in this notebook refer to that book.

**References:**
- Groves, P. D. (2013). *Principles of GNSS, Inertial, and Multi-Sensor Integrated Navigation Systems*, 2nd ed. Artech House. [ProQuest link](https://ebookcentral.proquest.com/lib/fh-ostschweiz/reader.action?docID=1531533&c=UERG&ppg=1)
- MathWorks: [Comparison of 3-D Coordinate Systems](https://ch.mathworks.com/help/map/choose-a-3-d-coordinate-system.html)

## 1) GNSS Output Data

GNSS data from a previous ARIS test flight is loaded here. The receiver is a u-blox ZED-F9P, which outputs the NMEA 0183 format containing WGS 84 coordinates (latitude, longitude, altitude above sea level).

In [ ]:

from pathlib import Path

import pandas as pd

base_path = Path("../../external/ARIS-flight-logs/Aris_Cargoflight_1_Logs")
location_path = base_path / "timeseries-location.csv"
velocity_path = base_path / "timeseries-velocity.csv"
location_df = pd.read_csv(location_path)
velocity_df = pd.read_csv(velocity_path)

df = pd.merge_asof(location_df, velocity_df, "timestamp [unix ms]")

# Convert lat/lon to numeric
df["location longitude [°]"] = pd.to_numeric(df["location longitude [°]"], errors="coerce")
df["location latitude [°]"] = pd.to_numeric(df["location latitude [°]"], errors="coerce")
df["location altitude [m above sea level]"] = pd.to_numeric(
    df["location altitude [m above sea level]"], errors="coerce"
)

# Drop rows where conversion failed
df = df.dropna(subset=[
    "location longitude [°]",
    "location latitude [°]",
    "location altitude [m above sea level]",
])

df.info()

In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp [unix ms]"], unit="ms")
df.dropna()
df.drop(axis=0, index=0, inplace=True)  # first value seems not initialized

df = df.sort_values("timestamp").reset_index(drop=True)
df.head(n=10)

## 2) Position Representations

Three coordinate representations appear throughout this pipeline:

**WGS 84 curvilinear position — $(L, \lambda, h)$**
Two angles and a height. Latitude $L$ is the angle between the equatorial plane and the normal to the WGS 84 ellipsoid at the point. Longitude $\lambda$ is the angle measured eastward from the IERS Reference Meridian. Height $h$ is the ellipsoidal height (the distance along the ellipsoid normal). This is the native output of GNSS receivers but due to the angle units it is unsuitable for vector math. It is not possible to meaningfully subtract two lat/lon pairs to get a distance in meters.

**Cartesian ECEF (earth-centered, earth-fixed) — $(x^e, y^e, z^e)$**
A 3D Cartesian system fixed to the Earth (meaning it rotates with the earth). Origin is at the Earth's center of mass. The $x^e$-axis points through the equator at 0° longitude, $y^e$ through the equator at 90° E, and $z^e$ along the rotation axis to the north pole. Because it's Cartesian, vector math with these values is straightforward.

**Local tangent-plane frame (l-frame, NED) — $(N, E, D)$**
A Cartesian frame with its origin fixed at a chosen point on the Earth's surface (for a rocket likely the launch pad or startup location). Axes point north, east, and down (NED) at that point. This is what the Kalman filter operates in, because:

- Position, velocity, and acceleration are all small numbers in meters, not huge ECEF values
- Dynamics are approximately linear over short distances
- Gravity is a simple constant vector $(0, 0, g)$
- It matches how we describe rocket trajectories

**Why not the local *navigation* frame (n-frame)?**
*"Position, velocity and acceleration referenced to a local navigation frame are meaningless"* (Chatper 2.5.2). This is because the n-frame's origin moves with the body, so position in it is always zero. For anything involving position, the l-frame with a fixed launch-pad origin is the correct choice.

The transformation pipeline used in this notebook is therefore:

$$
\underbrace{(L, \lambda, h)}_{\text{WGS 84}} \;\longrightarrow\; \underbrace{(x^e, y^e, z^e)}_{\text{ECEF}} \;\longrightarrow\; \underbrace{(N, E, D)}_{\text{local NED}}
$$

**A note on Groves' notation**
Throughout the book, vectors carry three indices: $r^{\gamma}_{\beta\alpha}$ means *"position of object $\alpha$, measured from origin $\beta$, expressed in the axes of frame $\gamma$."* So $r^e_{eb}$ is the rocket relative to Earth's center, in ECEF axes, while $r^l_{lb}$ is the rocket relative to the pad, in NED axes. Rotation matrices $C^l_e$ carry only two indices (no object origin) as they convert vectors from ECEF axes to l-frame axes.

The figure below shows the flight trajectory on a map using the raw WGS 84 coordinates, colored by altitude.

In [ ]:
import plotly.express as px

fig = px.scatter_map(
    df,
    lon="location longitude [°]",
    lat="location latitude [°]",
    color="location altitude [m above sea level]",
    color_continuous_scale=px.colors.sequential.Viridis,
    zoom=13.5,
    map_style="carto-positron"
)
fig.show()

A 3D view of the trajectory, with height visualized as vertical columns colored by altitude.

In [ ]:
import pydeck as pdk
import numpy as np

df_deck = df[
    [
        "location longitude [°]",
        "location latitude [°]",
        "location altitude [m above sea level]",
    ]
].iloc[::100].rename(columns={
    "location longitude [°]": "lon",
    "location latitude [°]": "lat",
    "location altitude [m above sea level]": "alt",
})
df_deck["alt"] = df_deck["alt"] - 408

alt_min, alt_max = df_deck["alt"].min(), df_deck["alt"].max()
t = (df_deck["alt"] - alt_min) / (alt_max - alt_min)
df_deck["r"] = np.clip(np.where(t < 0.5, 0, (t - 0.5) * 2) * 255, 0, 255).astype(int)
df_deck["g"] = np.clip(np.where(t < 0.5, t * 2, 1 - (t - 0.5) * 2) * 255, 0, 255).astype(int)
df_deck["b"] = np.clip(np.where(t < 0.5, 1 - t * 2, 0) * 255, 0, 255).astype(int)

points = df_deck[["lon", "lat"]].values.tolist()
view = pdk.data_utils.compute_view(points)
view.pitch = 75
view.bearing = 60

column_layer = pdk.Layer(
    "ColumnLayer",
    data=df_deck.to_dict(orient="records"),
    get_position=["lon", "lat"],
    get_elevation="alt",
    elevation_scale=1,
    radius=2,
    get_fill_color=["r", "g", "b"],
    pickable=False,
    auto_highlight=False,
)

r = pdk.Deck(
    column_layer,
    initial_view_state=view,
    map_provider="carto",
    map_style="dark",
)

r.to_html("gnss_visualization.html")

## 3) From GNSS to Local NED

A Kalman filter assumes approximately linear system dynamics and Gaussian noise. Curvilinear WGS 84 coordinates violate both assumptions. Latitude and longitude aren't linearly related to distance (a degree of longitude is ~111 km at the equator but shrinks to zero at the poles), and altitude is measured along a curved normal. Running an EKF directly on WGS 84 is theoretically possible but not recommended.

Converting to a local tangent-plane NED frame is the solution. The state becomes $(N, E, D)$ in meters, dynamics over the relevant flight range are linear.

### Assumptions

For the Kalman filter, the assumption of a flat tangent plane is valid as it has only minimal error in the position (approximately a few meters over a range of 20 km). Using this, the Kalman filter can be simplified a lot. Thus, the l-frame can be anchored once at the pad or startup location and never updated. It would be an option to do a reanchoring mid flight but this would mean shifting old NED positions to the new anchor point inside the running Kalman filter.

### The two transformations

**Step 1 — Geodetic to ECEF** (Groves eq. 2.112):

$$
\begin{aligned}
x^e_{eb} &= \bigl(R_E(L_b) + h_b\bigr)\cos L_b \cos \lambda_b \\
y^e_{eb} &= \bigl(R_E(L_b) + h_b\bigr)\cos L_b \sin \lambda_b \\
z^e_{eb} &= \bigl[(1 - e^2)R_E(L_b) + h_b\bigr]\sin L_b
\end{aligned}
$$

where $R_E(L)$ is the transverse radius of curvature (eq. 2.106):

$$
R_E(L) = \frac{R_0}{\sqrt{1 - e^2 \sin^2 L}}
$$

and the WGS 84 constants (Table 2.1) are $R_0 = 6{,}378{,}137.0$ m and $e = 0.0818191908425$.

**Step 2 — ECEF to local NED** (Groves eqs. 2.158 and 2.160):

First, build the rotation matrix $C^l_e$ from the **pad's** latitude $L_l$ and longitude $\lambda_l$. This is computed once at initialization:

$$
C^l_e = \begin{bmatrix}
-\sin L_l \cos \lambda_l & -\sin L_l \sin \lambda_l & \cos L_l \\
-\sin \lambda_l & \cos \lambda_l & 0 \\
-\cos L_l \cos \lambda_l & -\cos L_l \sin \lambda_l & -\sin L_l
\end{bmatrix}
$$

Then for every GNSS datapoint, apply:

$$
r^l_{lb} = C^l_e \bigl(r^e_{eb} - r^e_{el}\bigr)
$$

Read right to left: convert the rocket's WGS 84 fix to ECEF, subtract the pad's ECEF position (yielding a pad→rocket vector still in ECEF axes), then rotate into NED axes.

### Sensor frames (preview)

This notebook focuses on GNSS position only. When the IMU is integrated, it introduces a fourth frame: the **body frame (b-frame)**, whose axes are bolted to the rocket and rotate with it. IMU accelerations and angular rates are natively measured in the b-frame and must be rotated into the l-frame via the attitude matrix $C^l_b$ before they can be fused with GNSS position.

### Libraries

The implementation below is written from scratch for clarity and to match Groves' equations one-to-one. For usage in python, the [pymap3d](https://pypi.org/project/pymap3d/) library provides the same functionallity. A more heavyweight option would be [pyproj](https://pyproj4.github.io/pyproj/stable/).

### Implementation

The class below performs both steps of the pipeline. The pad location and rotation matrix are computed once via `set_anchor()`; each new GNSS fix is then converted with `update_position()`.

**Unit convention**: all latitude and longitude arguments are in **radians**, heights in **meters** (ellipsoidal).

In [ ]:
import math

EQUATORIAL_RADIUS: float = 6_378_137.0  # m (also denoted as R0)
ECCENTRICITY = 0.0818191908425  # (also denoted as e)


class NEDCoordinateFrame:
    def __init__(self):
        self.anchor = None
        self.transformation_matrix = None
        self.current_position_ned = None

    def set_anchor(self, lat, lon, alt):
        self.anchor = self._geodetic_to_ecef(lat, lon, alt)
        self.transformation_matrix = self._coordinate_transformation_matrix_ecef_to_ned(lat, lon, alt)

    def update_position(self, lat, lon, alt):
        if (self.anchor is None) or (self.transformation_matrix is None):
            raise ValueError("Anchor and transformation matrix must be set before updating position")

        current_position_ecef = self._geodetic_to_ecef(lat, lon, alt)
        delta_position_ecef = current_position_ecef - self.anchor
        self.current_position_ned = self.transformation_matrix @ delta_position_ecef
        return self.current_position_ned

    def _geodetic_to_ecef(self, lat, lon, alt):
        R_E = self._transverse_radius_of_curvature(lat)
        x = (R_E + alt) * math.cos(lat) * math.cos(lon)
        y = (R_E + alt) * math.cos(lat) * math.sin(lon)
        z = ((1 - ECCENTRICITY ** 2) * R_E + alt) * math.sin(lat)
        return np.array([x, y, z])

    def _coordinate_transformation_matrix_ecef_to_ned(self, lat, lon, alt):
        return np.array([[-math.sin(lat) * math.cos(lon), -math.sin(lat) * math.sin(lon), math.cos(lat)],
                         [-math.sin(lon), math.cos(lon), 0],
                         [-math.cos(lat) * math.cos(lon), -math.cos(lat) * math.sin(lon), -math.sin(lat)]])

    def _transverse_radius_of_curvature(self, lat: float):
        return EQUATORIAL_RADIUS / math.sqrt(1 - ECCENTRICITY ** 2 * math.sin(lat) ** 2)


In [ ]:
ned = NEDCoordinateFrame()
ned.set_anchor(math.radians(47.0), math.radians(8.0), 500.0)

# 1. Anchor maps to origin
print(ned.update_position(math.radians(47.0), math.radians(8.0), 500.0))

# 2. Pure altitude change
print(ned.update_position(math.radians(47.0), math.radians(8.0), 1500.0))

# 3. 0.001° north
print(ned.update_position(math.radians(47.001), math.radians(8.0), 500.0))

# 4. 0.001° east
print(ned.update_position(math.radians(47.0), math.radians(8.001), 500.0))

In [ ]:
import pymap3d as pm

# 1. Anchor maps to origin
print(pm.geodetic2ned(47.0, 8.0, 500.0, 47.0, 8.0, 500.0))

# 2. Pure altitude change
print(pm.geodetic2ned(47.0, 8.0, 1500.0, 47.0, 8.0, 500.0))

# 3. 0.001° north
print(pm.geodetic2ned(47.001, 8.0, 500.0, 47.0, 8.0, 500.0))

# 4. 0.001° east
print(pm.geodetic2ned(47.0, 8.001, 500.0, 47.0, 8.0, 500.0))

## 4) Next Steps: Velocity and Body-Frame Sensors

With the GNSS-to-NED position transform in place, the next pieces of the pipeline are:

- **Magnetometer**: tbd...
- **GNSS velocity → NED**: the receiver also outputs velocity in the ECEF frame, which rotates into the l-frame using the same $C^l_e$ (Groves eq. 2.152). No translation is needed because velocities are free vectors.
- **IMU body-frame data**: accelerations and angular rates come out in the b-frame and must be rotated into the l-frame using the attitude matrix $C^l_b$, which is maintained by integrating gyro measurements.
- **Attitude initialization**: on the pad, the rocket's orientation can be estimated from the accelerometer (which senses gravity pointing along $-x^b$ for a vertically-mounted rocket) and a magnetometer for heading.